In [9]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the project configuration
project_root = Path.cwd()

if not (project_root / "config/data_paths.json").exists():
    project_root = project_root.parent

with open(project_root / "config/data_paths.json") as f:
    config = json.load(f)

dataset_root = Path(config["dataset_root"])

# Select activity sessions (Rep1–Rep5)
activity_sessions = ["Rep1", "Rep2", "Rep3", "Rep4", "Rep5"]

processed_files = sorted(
    dataset_root.glob("*/P*/Processed Files/*.csv")
)

activity_files = [
    file for file in processed_files
    if file.stem.split("_P")[0] in activity_sessions
]

# Select chest-related sensor measurements
selected_columns = [
    "Time",
    "HR_Processed",
    "HRV",
    "Temperature",

    # ADAM 5 accelerometer (upper torso)
    "IMU_5_ax_g_",
    "IMU_5_ay_g_",
    "IMU_5_az_g_",

    # ADAM 5 gyroscope (upper torso)
    "IMU_5_gx_dps_",
    "IMU_5_gy_dps_",
    "IMU_5_gz_dps_",

    # Fatigue labels
    "Physical Fatigue - Initial",
    "Physical Fatigue - Final"
]

# Load five sample readings
sample_data = pd.read_csv(
    activity_files[0],
    usecols=selected_columns,
    nrows=5
)

print("Activity sessions:", len(activity_files))
print("Selected columns:", len(selected_columns))

display(sample_data)


Activity sessions: 335
Selected columns: 12


,Time,HR_Processed,HRV,Temperature,IMU_5_ax_g_,IMU_5_ay_g_,IMU_5_az_g_,IMU_5_gx_dps_,IMU_5_gy_dps_,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final
0,522500,102.5,22.402814,34.224213,0.042525,0.129985,0.011618,14.436704,14.436704,14.436704,1,3
1,523000,104.7,22.376775,34.226817,-0.061162,-2.334652,0.173983,1.233281,1.233281,1.233281,1,3
2,523500,108.2,22.351200,34.228885,0.348357,-2.585750,0.150009,13.288418,13.288418,13.288418,1,3
3,524000,108.2,22.326121,34.230427,0.020237,-0.042484,0.012974,-34.480159,-34.480159,-34.480159,1,3
4,524500,108.2,22.301569,34.231472,0.039516,-0.311922,0.089224,-5.812640,-5.812640,-5.812640,1,3



## Check Gyroscope Data


In [10]:

gyro_columns = [
    "IMU_5_gx_dps_",
    "IMU_5_gy_dps_",
    "IMU_5_gz_dps_"
]

total_rows = 0
identical_rows = 0
identical_sessions = 0

for file in activity_files:
    data = pd.read_csv(file, usecols=gyro_columns)

    # Check whether X, Y, and Z have identical readings
    same = (
        (data[gyro_columns[0]] == data[gyro_columns[1]]) &
        (data[gyro_columns[1]] == data[gyro_columns[2]])
    )

    total_rows += len(data)
    identical_rows += same.sum()

    if same.all():
        identical_sessions += 1

print("Total readings:", total_rows)
print("Identical readings:", identical_rows)
print("Identical readings (%):", round(identical_rows / total_rows * 100, 2))
print("Sessions with identical axes:", identical_sessions, "/", len(activity_files))


Total readings: 286840
Identical readings: 286840
Identical readings (%): 100.0
Sessions with identical axes: 335 / 335



### Combine Activity Sessions


In [11]:

# Combine sensor readings from all 335 activity CSV files
combined_sessions = []

for file in activity_files:
    data = pd.read_csv(
        file,
        usecols=selected_columns
    )

    # Identify the task, participant, and repetition
    parts = file.relative_to(dataset_root).parts

    data.insert(0, "Task", parts[0])
    data.insert(1, "Participant", parts[1])
    data.insert(2, "Session", file.stem.split("_P")[0])

    combined_sessions.append(data)

# Create one combined dataset
combined_df = pd.concat(
    combined_sessions,
    ignore_index=True
)

# Verify the combined dataset
print("Total rows:", len(combined_df))
print("Total columns:", len(combined_df.columns))
print("Unique participants:", combined_df["Participant"].nunique())
print("Activity sessions:", len(activity_files))

print("\nRows by task:")
print(combined_df["Task"].value_counts())

display(combined_df.head())


Total rows: 286840
Total columns: 15
Unique participants: 38
Activity sessions: 335

Rows by task:
Task
Wire Harness    147624
Composite       139216
Name: count, dtype: int64


,Task,Participant,Session,Time,HR_Processed,HRV,Temperature,IMU_5_ax_g_,IMU_5_ay_g_,IMU_5_az_g_,IMU_5_gx_dps_,IMU_5_gy_dps_,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final
0,Composite,P1,Rep1,522500,102.5,22.402814,34.224213,0.042525,0.129985,0.011618,14.436704,14.436704,14.436704,1,3
1,Composite,P1,Rep1,523000,104.7,22.376775,34.226817,-0.061162,-2.334652,0.173983,1.233281,1.233281,1.233281,1,3
2,Composite,P1,Rep1,523500,108.2,22.351200,34.228885,0.348357,-2.585750,0.150009,13.288418,13.288418,13.288418,1,3
3,Composite,P1,Rep1,524000,108.2,22.326121,34.230427,0.020237,-0.042484,0.012974,-34.480159,-34.480159,-34.480159,1,3
4,Composite,P1,Rep1,524500,108.2,22.301569,34.231472,0.039516,-0.311922,0.089224,-5.812640,-5.812640,-5.812640,1,3


### Check Invalid Sensor Values


In [12]:

# Check missing values across all sensor columns
sensor_columns = [
    "HR_Processed", "HRV", "Temperature",
    "IMU_5_ax_g_", "IMU_5_ay_g_", "IMU_5_az_g_",
    "IMU_5_gx_dps_", "IMU_5_gy_dps_", "IMU_5_gz_dps_"
]

quality_df = pd.DataFrame({
    "Missing Values": combined_df[sensor_columns].isna().sum(),
    "Minimum": combined_df[sensor_columns].min(),
    "Maximum": combined_df[sensor_columns].max()
})

# Count physically invalid HR and HRV readings
invalid_hr = (combined_df["HR_Processed"] <= 0).sum()
invalid_hrv = (combined_df["HRV"] < 0).sum()

print("Total rows:", len(combined_df))
print("Invalid HR readings:", invalid_hr)
print("Negative HRV readings:", invalid_hrv)

display(quality_df)


Total rows: 286840
Invalid HR readings: 52586
Negative HRV readings: 4836


,Missing Values,Minimum,Maximum
HR_Processed,0,-3.000000e+00,2.324000e+02
HRV,0,-3.543623e+07,1.503864e+08
Temperature,0,2.071338e+01,3.939159e+01
IMU_5_ax_g_,0,-2.901668e+00,3.290189e+00
IMU_5_ay_g_,0,-2.890582e+00,4.787902e+00
IMU_5_az_g_,0,-2.997957e+00,3.303338e+00
IMU_5_gx_dps_,0,-1.146740e+03,5.921526e+02
IMU_5_gy_dps_,0,-1.146740e+03,5.921526e+02
IMU_5_gz_dps_,0,-1.146740e+03,5.921526e+02



### Investigate Invalid HRV Readings


In [13]:

# Use original HRV data without modifying any values
hrv = combined_df["HRV"]

# Count negative, zero, and extremely large values
print("Total HRV readings:", len(hrv))
print("Negative HRV readings:", (hrv < 0).sum())
print("Zero HRV readings:", (hrv == 0).sum())
print("HRV readings above 10000:", (hrv > 10000).sum())

print("\nHRV Minimum:", hrv.min())
print("HRV Maximum:", hrv.max())

# Examine the distribution of HRV values
print("\nHRV Distribution:")
display(
    hrv.quantile([0, 0.01, 0.25, 0.50, 0.75, 0.99, 1])
    .to_frame("HRV")
)

# Inspect actual negative HRV measurements
columns = ["Task", "Participant", "Session", "Time", "HRV"]

print("\nExamples of Negative HRV:")
display(combined_df.loc[hrv < 0, columns].head(10))

# Inspect unusually large HRV measurements
print("\nExamples of Extremely Large HRV:")
display(combined_df.loc[hrv > 10000, columns].head(10))


Total HRV readings: 286840
Negative HRV readings: 4836
Zero HRV readings: 0
HRV readings above 10000: 5685

HRV Minimum: -35436228.6380984
HRV Maximum: 150386406.432426

HRV Distribution:


,HRV
0.00,-3.543623e+07
0.01,-2.162651e+03
0.25,3.150278e+01
0.50,6.001850e+01
0.75,1.530255e+02
0.99,3.976237e+04
1.00,1.503864e+08



Examples of Negative HRV:


,Task,Participant,Session,Time,HRV
19165,Composite,P13,Rep3,1287500,-0.026758
19166,Composite,P13,Rep3,1288000,-0.113365
19167,Composite,P13,Rep3,1288500,-0.188085
19168,Composite,P13,Rep3,1289000,-0.250861
19169,Composite,P13,Rep3,1289500,-0.301633
19170,Composite,P13,Rep3,1290000,-0.340344
19171,Composite,P13,Rep3,1290500,-0.366936
19172,Composite,P13,Rep3,1291000,-0.381349
19173,Composite,P13,Rep3,1291500,-0.383527
19174,Composite,P13,Rep3,1292000,-0.373410



Examples of Extremely Large HRV:


,Task,Participant,Session,Time,HRV
83579,Composite,P36,Rep1,360500,274496.126732
83580,Composite,P36,Rep1,361000,276234.293276
83581,Composite,P36,Rep1,361500,277933.145279
83582,Composite,P36,Rep1,362000,279591.598857
83583,Composite,P36,Rep1,362500,281208.570127
83584,Composite,P36,Rep1,363000,282782.975205
83585,Composite,P36,Rep1,363500,284313.730208
83586,Composite,P36,Rep1,364000,285799.751251
83587,Composite,P36,Rep1,364500,287239.954451
83588,Composite,P36,Rep1,365000,288633.255924



### Handle Invalid HR and HRV Values


In [14]:

# Create a copy to preserve the original dataset
cleaned_df = combined_df.copy()

# Identify invalid sensor readings
invalid_hr = cleaned_df["HR_Processed"] <= 0
invalid_hrv = cleaned_df["HRV"] < 0

# Count invalid readings
print("Invalid HR readings:", invalid_hr.sum())
print("Invalid HRV readings:", invalid_hrv.sum())

# Replace invalid readings with NaN
cleaned_df.loc[invalid_hr, "HR_Processed"] = np.nan
cleaned_df.loc[invalid_hrv, "HRV"] = np.nan

# Verify the cleaning results
print("\nTotal rows before:", len(combined_df))
print("Total rows after:", len(cleaned_df))

print("\nMissing values after cleaning:")
display(
    cleaned_df[["HR_Processed", "HRV"]]
    .isna()
    .sum()
    .to_frame("Missing Values")
)


Invalid HR readings: 52586
Invalid HRV readings: 4836

Total rows before: 286840
Total rows after: 286840

Missing values after cleaning:


,Missing Values
HR_Processed,52586
HRV,4836



### Check Valid Sensor Readings per Session


In [15]:

# Count remaining valid HR and HRV readings in each activity session
session_quality = cleaned_df.groupby(
    ["Task", "Participant", "Session"]
).agg(
    Total_Readings=("Time", "size"),
    Valid_HR=("HR_Processed", "count"),
    Valid_HRV=("HRV", "count")
).reset_index()

# Calculate percentage of valid readings
session_quality["HR_Valid_Percent"] = (
    session_quality["Valid_HR"] /
    session_quality["Total_Readings"] * 100
).round(2)

session_quality["HRV_Valid_Percent"] = (
    session_quality["Valid_HRV"] /
    session_quality["Total_Readings"] * 100
).round(2)

print("Total sessions:", len(session_quality))
print("Sessions with no valid HR:", (session_quality["Valid_HR"] == 0).sum())
print("Sessions with no valid HRV:", (session_quality["Valid_HRV"] == 0).sum())

display(session_quality.head(10))


Total sessions: 335
Sessions with no valid HR: 6
Sessions with no valid HRV: 0


,Task,Participant,Session,Total_Readings,Valid_HR,Valid_HRV,HR_Valid_Percent,HRV_Valid_Percent
0,Composite,P1,Rep1,1029,977,1029,94.95,100.0
1,Composite,P1,Rep2,867,867,867,100.00,100.0
2,Composite,P1,Rep3,826,826,826,100.00,100.0
3,Composite,P1,Rep4,788,788,788,100.00,100.0
4,Composite,P1,Rep5,697,697,697,100.00,100.0
5,Composite,P10,Rep1,1567,1567,1567,100.00,100.0
6,Composite,P10,Rep2,1104,1104,1104,100.00,100.0
7,Composite,P10,Rep3,1429,1429,1429,100.00,100.0
8,Composite,P10,Rep4,965,965,965,100.00,100.0
9,Composite,P10,Rep5,945,945,945,100.00,100.0



### Identify Sessions Without Valid HR


In [16]:

# Find activity sessions containing no valid HR readings
missing_hr_sessions = session_quality[
    session_quality["Valid_HR"] == 0
]

print("Sessions with no valid HR:", len(missing_hr_sessions))

display(missing_hr_sessions)


Sessions with no valid HR: 6


,Task,Participant,Session,Total_Readings,Valid_HR,Valid_HRV,HR_Valid_Percent,HRV_Valid_Percent
95,Composite,P31,Rep1,2,0,2,0.0,100.00
196,Wire Harness,P14,Rep2,587,0,352,0.0,59.97
198,Wire Harness,P14,Rep4,689,0,563,0.0,81.71
209,Wire Harness,P16,Rep5,972,0,972,0.0,100.00
277,Wire Harness,P37,Rep3,1212,0,1212,0.0,100.00
278,Wire Harness,P37,Rep4,1133,0,1133,0.0,100.00



### Investigate P31 Rep1 Session


In [17]:

# Select Composite, Participant P31, Rep1
p31_rep1 = combined_df[
    (combined_df["Task"] == "Composite") &
    (combined_df["Participant"] == "P31") &
    (combined_df["Session"] == "Rep1")
].copy()

# Calculate the time covered by the readings
duration_ms = (
    p31_rep1["Time"].max() - p31_rep1["Time"].min()
)

print("Total readings:", len(p31_rep1))
print("Start time (ms):", p31_rep1["Time"].min())
print("End time (ms):", p31_rep1["Time"].max())
print("Recorded duration (seconds):", duration_ms / 1000)

display(p31_rep1)


Total readings: 2
Start time (ms): 2000
End time (ms): 2500
Recorded duration (seconds): 0.5


,Task,Participant,Session,Time,HR_Processed,HRV,Temperature,IMU_5_ax_g_,IMU_5_ay_g_,IMU_5_az_g_,IMU_5_gx_dps_,IMU_5_gy_dps_,IMU_5_gz_dps_,Physical Fatigue - Initial,Physical Fatigue - Final
70824,Composite,P31,Rep1,2000,-2.0,4823.922265,32.434108,-0.206027,0.883237,0.475285,7.563023,7.563023,7.563023,0,2
70825,Composite,P31,Rep1,2500,-2.0,4734.798136,32.419974,-0.175211,0.886302,0.487524,8.469278,8.469278,8.469278,0,2



### Check Original HR for P31 Rep1


In [18]:

# Find the original CSV for Composite P31 Rep1
file = next(
    f for f in activity_files
    if f.relative_to(dataset_root).parts[0] == "Composite"
    and f.relative_to(dataset_root).parts[1] == "P31"
    and f.stem.split("_P")[0] == "Rep1"
)

# Compare original and processed heart rate
original_hr = pd.read_csv(
    file,
    usecols=["Time", "HR", "HR_Processed"]
)

print("Total readings:", len(original_hr))
display(original_hr)


Total readings: 2


,Time,HR_Processed,HR
0,2000,-2,12.292313
1,2500,-2,12.799194



### Check Original HR in Remaining Sessions


In [19]:

results = []

# Check original HR values in sessions without valid processed HR
for file in activity_files:
    parts = file.relative_to(dataset_root).parts
    task, participant = parts[0], parts[1]
    session = file.stem.split("_P")[0]

    if not (
        (missing_hr_sessions["Task"] == task) &
        (missing_hr_sessions["Participant"] == participant) &
        (missing_hr_sessions["Session"] == session)
    ).any():
        continue

    data = pd.read_csv(file, usecols=["HR"])

    results.append({
        "Task": task,
        "Participant": participant,
        "Session": session,
        "Readings": len(data),
        "HR Minimum": data["HR"].min(),
        "HR Median": data["HR"].median(),
        "HR Maximum": data["HR"].max()
    })

display(pd.DataFrame(results))


,Task,Participant,Session,Readings,HR Minimum,HR Median,HR Maximum
0,Composite,P31,Rep1,2,12.292313,12.545753,12.799194
1,Wire Harness,P14,Rep2,587,86.364386,95.657052,109.291738
2,Wire Harness,P14,Rep4,689,7.408207,91.500805,92.678924
3,Wire Harness,P16,Rep5,972,92.630450,106.044604,119.299628
4,Wire Harness,P37,Rep3,1212,57.240091,75.371287,117.705864
5,Wire Harness,P37,Rep4,1133,42.806129,95.038420,140.460163



### Remove Insufficient Activity Session


In [20]:

# Identify the session containing only two readings
remove_session = (
    (cleaned_df["Task"] == "Composite") &
    (cleaned_df["Participant"] == "P31") &
    (cleaned_df["Session"] == "Rep1")
)

# Remove this session from the cleaned dataset only
cleaned_df = cleaned_df.loc[~remove_session].copy()

print("Remaining rows:", len(cleaned_df))
print(
    "Remaining sessions:",
    cleaned_df[["Task", "Participant", "Session"]]
    .drop_duplicates().shape[0]
)


Remaining rows: 286838
Remaining sessions: 334



## Prepare Session Time Data


In [21]:

# Sort sensor readings chronologically within each session
group_columns = ["Task", "Participant", "Session"]

cleaned_df = cleaned_df.sort_values(
    group_columns + ["Time"]
).reset_index(drop=True)

# Calculate elapsed time from the start of each session
session_start = cleaned_df.groupby(
    group_columns
)["Time"].transform("min")

cleaned_df["Elapsed_Time_s"] = (
    cleaned_df["Time"] - session_start
) / 1000

# Check the results
print("Total sessions:", cleaned_df[group_columns].drop_duplicates().shape[0])

display(
    cleaned_df[
        group_columns + ["Time", "Elapsed_Time_s"]
    ].head(10)
)


Total sessions: 334


,Task,Participant,Session,Time,Elapsed_Time_s
0,Composite,P1,Rep1,522500,0.0
1,Composite,P1,Rep1,523000,0.5
2,Composite,P1,Rep1,523500,1.0
3,Composite,P1,Rep1,524000,1.5
4,Composite,P1,Rep1,524500,2.0
5,Composite,P1,Rep1,525000,2.5
6,Composite,P1,Rep1,525500,3.0
7,Composite,P1,Rep1,526000,3.5
8,Composite,P1,Rep1,526500,4.0
9,Composite,P1,Rep1,527000,4.5



### Save Preprocessed Dataset


In [22]:

# Create the output directory
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Save the cleaned sensor data
output_file = output_dir / "cleaned_activity_sessions.csv"

cleaned_df.to_csv(output_file, index=False)

# Verify the saved dataset
print("Saved to:", output_file)
print("Total rows:", len(cleaned_df))
print("Total columns:", len(cleaned_df.columns))
print(
    "Total sessions:",
    cleaned_df[["Task", "Participant", "Session"]]
    .drop_duplicates().shape[0]
)


Saved to: c:\Users\WINSYS\Desktop\Physical_Fatigue_ML\data\processed\cleaned_activity_sessions.csv
Total rows: 286838
Total columns: 16
Total sessions: 334
